# Kohonen: sequential hyperparameter tuning

Each stage sweeps one hyperparameter with the others frozen at the best values so far. Every candidate is scored with 5-fold cross-validation x 3 seeds: the SOM is trained on 80% of the creators and evaluated on the 20% it has not seen. Generated by `experiments/run_kohonen_tuning.py`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_dir = Path.cwd().resolve()
if not (project_dir / "src").is_dir():
    project_dir = project_dir.parent
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.tp4.kohonen.visualization import visualize_som_weights, visualize_u_matrix

results_dir = project_dir / "results"
models_dir = results_dir / "kohonen_models"

tuning = pd.read_csv(results_dir / "kohonen_tuning.csv")
with open(results_dir / "kohonen_best_config.json", encoding="utf-8") as f:
    best = json.load(f)

print("Final config:", json.dumps(best["config"], indent=2))

## Stage summary

Mean over folds and seeds. **heldout_qe** ranks the candidates; **heldout_te** (topographic error) is reported next to it; **train_qe** shows how much the map memorises.

In [ ]:
stage_tables = {}
for (stage, param), g in tuning.groupby(["stage", "param"], sort=True):
    stage_tables[param] = (
        g.groupby("value")
        .agg(heldout_qe=("heldout_qe", "mean"), heldout_qe_std=("heldout_qe", "std"),
             heldout_te=("heldout_te", "mean"), train_qe=("train_qe", "mean"))
        .sort_values("heldout_qe")
    )
    print(f"Stage {stage}: {param}")
    display(stage_tables[param].round(4))

## Effect of each hyperparameter

Points are individual runs (fold x seed), the line is the mean. The red circle is the stage winner (chosen on held-out QE). Differences smaller than the spread of the points are not meaningful.

In [ ]:
stages = tuning[["stage", "param"]].drop_duplicates().sort_values("stage")
winners = {w["param"]: w["winner"] for w in best["winners"]}

metrics = [("heldout_qe", "Held-out QE"), ("heldout_te", "Held-out topographic error")]
fig, axes = plt.subplots(2, len(stages), figsize=(4 * len(stages), 7), squeeze=False)
for col, (_, row) in enumerate(stages.iterrows()):
    param = row["param"]
    g = tuning[tuning["param"] == param].copy()
    g["value"] = g["value"].astype(str)
    order = list(dict.fromkeys(g["value"]))
    for r, (metric, label) in enumerate(metrics):
        ax = axes[r, col]
        sns.stripplot(data=g, x="value", y=metric, order=order, color="gray", alpha=0.4, size=3, ax=ax)
        mean = g.groupby("value")[metric].mean().loc[order]
        ax.plot(range(len(order)), mean.values, color="steelblue", marker="o")
        win = str(winners[param])
        ax.plot(order.index(win), mean[win], "o", color="red", markersize=12, mfc="none", mew=2)
        ax.set_xlabel(param)
        ax.set_ylabel(label)
        if r == 0:
            ax.set_title(f"Stage {int(row['stage'])}: {param}")
plt.tight_layout()
plt.show()

## Neuron usage per stage

In [ ]:
usage = tuning.assign(usage_ratio=tuning["used_neurons"] / tuning["total_neurons"])
display(usage.groupby(["param", "value"])["usage_ratio"].mean().round(3).to_frame("mean usage ratio"))

## Final model

In [ ]:
with np.load(models_dir / "som_tuned.npz") as saved:
    weights = saved["weights"]
    bmu_counts = saved["bmu_counts"]
    u_matrix = saved["u_matrix"]
    feature_names = saved["feature_names"].tolist()
with open(models_dir / "som_tuned.json", encoding="utf-8") as f:
    metadata = json.load(f)
grid_type = metadata["grid_type"]
print(json.dumps(metadata, indent=2))

In [ ]:
for feature in range(min(2, weights.shape[2])):
    print(f"Feature {feature}: {feature_names[feature]}")
    visualize_som_weights(weights, feature=feature, grid_type=grid_type)

In [ ]:
visualize_u_matrix(u_matrix, grid_type=grid_type)

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(bmu_counts, annot=True, fmt="d", cmap="Blues")
plt.title("Creators per neuron (rectangular layout; hexagonal offsets not drawn)")
plt.xlabel("Neuron column")
plt.ylabel("Neuron row")
plt.tight_layout()
plt.show()